In [2]:
import sys
import os

# Detect environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

# Environment-specific settings
if IS_COLAB:
    print("Running on Google Colab")
    ENV_NAME = "colab"
    USE_GPU = True
    TESSERACT_PATH = None  
    
elif IS_KAGGLE:
    print("Running on Kaggle")
    ENV_NAME = "kaggle"
    USE_GPU = True
    TESSERACT_PATH = None  
    
else:  # Local
    print("Running Locally")
    ENV_NAME = "local"
    USE_GPU = False  
    TESSERACT_PATH = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

print(f"Environment: {ENV_NAME}")
print(f"GPU enabled: {USE_GPU}")

Running Locally
Environment: local
GPU enabled: False


# Environment Configuration

Auto-detect running environment (Local, Kaggle, or Google Colab)

# Installation

In [3]:
# Fix NumPy version compatibility
!pip install -q "numpy<2.0"

!pip install -q pymupdf layoutparser torch torchvision
!pip install -q "detectron2@git+https://github.com/facebookresearch/detectron2.git@v0.6"
!pip install -q "git+https://github.com/Layout-Parser/layout-parser.git"

# Fix Pillow version for compatibility with torchvision
!pip install -q "Pillow>=10.0.0"

# OCR engines - only VietOCR
!pip install -q vietocr

# PhoBERT dependencies
!pip install -q transformers
!pip install -q scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
doclayout-yolo 0.0.3 requires albumentations>=1.4.11, but you have albumentations 1.4.2 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [350 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      

# Libraries

In [1]:
import os
import fitz  # PyMuPDF
import cv2
import re
import requests
import json
from urllib.parse import urlparse
from datetime import datetime
import numpy as np
from PIL import Image
import pytesseract

# OCR engines - VietOCR only
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

import torch
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity


# Initialize Models

In [2]:
if 'USE_GPU' not in globals():
    USE_GPU = False
    TESSERACT_PATH = None

# Set Tesseract path for Windows
if TESSERACT_PATH:
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH
    print(f"Tesseract path set: {TESSERACT_PATH}")
else:
    # Try default Windows path
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Initialize VietOCR (for Vietnamese text recognition)
print("Loading VietOCR...")
try:
    if 'viet_ocr' not in globals():
        config = Cfg.load_config_from_name('vgg_transformer')
        import torch
        device_vietocr = 'cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu'
        config['device'] = device_vietocr
        config['predictor']['beamsearch'] = False
        viet_ocr = Predictor(config)
        print(f"VietOCR loaded on {device_vietocr}")
    else:
        print(f"VietOCR already loaded (reusing existing instance)")
except Exception as e:
    print(f"VietOCR error: {type(e).__name__}: {str(e)}")
    raise

# Initialize PhoBERT
print("Loading PhoBERT...")
try:
    if 'phobert' not in globals():
        import torch
        device = torch.device('cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu')
        phobert = AutoModel.from_pretrained("vinai/phobert-base")
        tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
        phobert.eval()
        phobert.to(device)
        print(f"PhoBERT loaded on {device}")
    else:
        print(f"PhoBERT already loaded (reusing existing instance)")
except Exception as e:
    print(f"PhoBERT error: {type(e).__name__}: {str(e)}")
    raise

Loading VietOCR...


c:\Users\Xuan Nhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model weight C:\Users\XUANNH~1\AppData\Local\Temp\vgg_transformer.pth exsits. Ignore download!
VietOCR loaded on cpu
Loading PhoBERT...
PhoBERT loaded on cpu


# PhoBERT Processing Functions

In [3]:
def get_text_embedding(text, max_length=256):
    """
    Encode Vietnamese text using PhoBERT
    Returns sentence embedding (average of token embeddings)
    """
    if not text or len(text.strip()) == 0:
        return np.zeros((1, 768))  # PhoBERT embedding size
    
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=max_length,
        truncation=True,
        padding='max_length'
    ).to(device)
    
    # Get embeddings
    with torch.no_grad():
        outputs = phobert(**inputs)
        # Use mean pooling over token embeddings
        embeddings = outputs.last_hidden_state.mean(dim=1)
    
    return embeddings.cpu().numpy()


def calculate_similarity(text1, text2):
    """
    Calculate cosine similarity between two Vietnamese texts
    """
    emb1 = get_text_embedding(text1)
    emb2 = get_text_embedding(text2)
    
    similarity = cosine_similarity(emb1, emb2)[0][0]
    return float(similarity)


def classify_problem_type(question_text):
    """
    Classify problem type using PhoBERT semantic similarity
    Returns: (type, confidence_scores)
    """
    # Define problem type keywords
    problem_types = {
        'geometry': 'hình học tam giác tứ giác đường tròn góc đoạn thẳng chu vi diện tích',
        'algebra': 'phương trình bất phương trình hệ phương trình biến số nghiệm',
        'number_theory': 'số nguyên chia hết ước số bội số số chính phương',
        'measurement': 'đo lường đơn vị chuyển đổi tính toán',
        'word_problem': 'bài toán thực tế tính toán ứng dụng'
    }
    
    # Get embedding for question
    question_emb = get_text_embedding(question_text)
    
    # Compare with each type
    scores = {}
    for ptype, keywords in problem_types.items():
        type_emb = get_text_embedding(keywords)
        similarity = cosine_similarity(question_emb, type_emb)[0][0]
        scores[ptype] = float(similarity)
    
    # Get best match
    best_type = max(scores, key=scores.get)
    
    return best_type, scores


def analyze_figure_relevance(question_text, figure_refs):
    # Keywords that suggest figure importance
    figure_keywords = [
        'trong hình', 'hình vẽ', 'theo hình', 'quan sát hình',
        'từ hình', 'dựa vào hình', 'xem hình', 'cho hình'
    ]
    
    # Check if question explicitly mentions figures
    explicit_mention = any(keyword in question_text.lower() for keyword in figure_keywords)
    
    # Check if figure references appear in question
    has_figure_refs = len(figure_refs) > 0
    
    # Calculate semantic relevance
    figure_context = "hình vẽ minh họa đồ thị biểu đồ sơ đồ"
    semantic_score = calculate_similarity(question_text, figure_context)
    
    # Combine factors
    relevance = (
        (0.4 if explicit_mention else 0) +
        (0.3 if has_figure_refs else 0) +
        (0.3 * semantic_score)
    )
    
    return min(relevance, 1.0)

# Utils - PDF Processing

In [4]:
def download_pdf_from_url(url, save_dir="input", chunk_size=8192):
    os.makedirs(save_dir, exist_ok=True)
    filename = os.path.basename(urlparse(url).path)
    if not filename.endswith(".pdf"):
        filename = "document.pdf"

    save_path = os.path.join(save_dir, filename)
    if os.path.exists(save_path):
        print(f"PDF already exists: {save_path}")
        return save_path

    print(f"Download PDF...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size):
                if chunk:
                    f.write(chunk)
    
    print(f"Saved: {save_path}")
    return save_path


def pdf_to_images(pdf_path, out_dir="pdf_pages", dpi=200):
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)

    image_paths = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img_path = f"{out_dir}/page_{i+1:03d}.png"
        pix.save(img_path)
        image_paths.append(img_path)

    print(f"Converted {len(image_paths)} pages to images")
    return image_paths

# OCR Processing

In [17]:
def ocr_page_hybrid_tesseract_vietocr(image_path):
    """
    HYBRID APPROACH: Tesseract detects text regions, VietOCR recognizes Vietnamese
    - Tesseract: Good at detecting text layout and bounding boxes
    - VietOCR: Better at recognizing Vietnamese characters correctly
    """
    try:
        # Load image
        img = Image.open(image_path)
        img_cv = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        
        # Step 1: Use Tesseract to detect text regions (get bounding boxes)
        # Output_type=Output.DICT returns dict with bbox coordinates
        ocr_data = pytesseract.image_to_data(img, lang='vie', output_type=pytesseract.Output.DICT)
        
        # Step 2: Extract valid text boxes
        text_blocks = []
        n_boxes = len(ocr_data['text'])
        
        for i in range(n_boxes):
            # Filter out empty or low-confidence detections
            conf = int(ocr_data['conf'][i]) if ocr_data['conf'][i] != '-1' else 0
            text = ocr_data['text'][i].strip()
            
            # Skip if confidence too low or empty
            if conf < 30 or len(text) == 0:
                continue
            
            # Get bounding box from Tesseract
            x = ocr_data['left'][i]
            y = ocr_data['top'][i]
            w = ocr_data['width'][i]
            h = ocr_data['height'][i]
            
            # Skip very small boxes
            if w < 20 or h < 10:
                continue
            
            # Add padding
            padding = 5
            x = max(0, x - padding)
            y = max(0, y - padding)
            w = min(img_cv.shape[1] - x, w + 2*padding)
            h = min(img_cv.shape[0] - y, h + 2*padding)
            
            # Crop region
            cropped = img_rgb[y:y+h, x:x+w]
            
            if cropped.size == 0:
                continue
            
            # Step 3: Use VietOCR to recognize Vietnamese text in cropped region
            try:
                cropped_pil = Image.fromarray(cropped)
                vietocr_text = viet_ocr.predict(cropped_pil)
                
                if vietocr_text and len(vietocr_text.strip()) > 0:
                    text_blocks.append({
                        'text': vietocr_text,
                        'y': y,
                        'x': x,
                        'line_num': ocr_data['line_num'][i],
                        'block_num': ocr_data['block_num'][i]
                    })
            except Exception as e:
                # Fallback to Tesseract text if VietOCR fails
                text_blocks.append({
                    'text': text,
                    'y': y,
                    'x': x,
                    'line_num': ocr_data['line_num'][i],
                    'block_num': ocr_data['block_num'][i]
                })
        
        # Step 4: Sort by block, line, then x position (reading order)
        text_blocks.sort(key=lambda b: (b['block_num'], b['line_num'], b['x']))
        
        # Step 5: Merge text with proper spacing and line breaks
        final_text = []
        prev_block = None
        prev_line = None
        
        for block in text_blocks:
            # Add paragraph break between blocks
            if prev_block is not None and block['block_num'] != prev_block:
                final_text.append('\n\n')
            # Add line break between lines in same block
            elif prev_line is not None and block['line_num'] != prev_line:
                final_text.append('\n')
            # Add space between words on same line
            elif len(final_text) > 0 and final_text[-1] not in ['\n', '\n\n']:
                final_text.append(' ')
            
            final_text.append(block['text'])
            prev_block = block['block_num']
            prev_line = block['line_num']
        
        full_text = ''.join(final_text)
        
        # Clean up whitespace
        full_text = re.sub(r'[ \t]+', ' ', full_text)
        full_text = re.sub(r'\n\s+\n', '\n\n', full_text)
        
        return full_text
        
    except Exception as e:
        print(f"Hybrid OCR error: {e}")
        return ""


def ocr_page_with_tesseract(image_path):
    """
    OCR text from image using Tesseract only (for comparison)
    """
    try:
        img = Image.open(image_path)
        custom_config = r'--oem 3 --psm 3'
        text = pytesseract.image_to_string(img, lang='vie', config=custom_config)
        
        # Clean up whitespace
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n\s+\n', '\n\n', text)
        
        return text
    except Exception as e:
        print(f"Tesseract error: {e}")
        return ""


# Problem Extraction with PhoBERT

In [18]:
def extract_problems_with_figures_phobert(text, min_figure_relevance=0.0):
    """
    Extract problems using PhoBERT for better Vietnamese understanding
    Enhanced version with semantic analysis
    
    Args:
        text: OCR text to extract problems from
        min_figure_relevance: Minimum figure relevance score (0.0-1.0) to include problem.
                             Set to 0.0 to extract all problems, then filter later.
    """
    figure_patterns = [
        (r'Hình\s*\d+\.\d+', 'Hình X.Y'),
        (r'\(H\.\d+\.\d+\)', '(H.X.Y)'),
        (r'trong\s+(?:các\s+)?hình\s+(?:vẽ\s+)?(?:sau|trên|dưới|bên)', 'trong hình...'),
    ]
    
    # Keywords to filter out chart/graph problems
    chart_keywords = [
        r'biểu\s*đồ',
        r'đồ\s*thị',
        r'chart',
        r'graph',
        r'bảng\s*thống\s*kê',
    ]
    
    results = []
    all_markers = []
    
    # Find numbered problems
    for match in re.finditer(r'(?:^|\n|(?<=\s{2}))(\d+\.\d+)\.?\s*', text, re.MULTILINE):
        problem_num = match.group(1)
        
        if match.start() > 0:
            before = text[max(0, match.start()-5):match.start()]
            if re.search(r'[a-zđáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵ]\s*$', before, re.IGNORECASE):
                continue
        
        all_markers.append({
            'type': 'numbered',
            'identifier': problem_num,
            'start': match.start(),
            'end': match.end(),
            'full_match': match.group(0).strip()
        })
    
    # Find example problems
    for match in re.finditer(r'Ví\s+dụ\s+(\d+)', text, re.IGNORECASE):
        all_markers.append({
            'type': 'example',
            'identifier': f'VD{match.group(1)}',
            'start': match.start(),
            'end': match.end(),
            'full_match': match.group(0)
        })
    
    # Sort and deduplicate markers
    all_markers.sort(key=lambda x: x['start'])
    unique_markers = []
    last_pos = -100
    for marker in all_markers:
        if marker['start'] - last_pos > 5:
            unique_markers.append(marker)
            last_pos = marker['start']
    
    all_markers = unique_markers
    
    if not all_markers:
        return results
    
    # Track figures seen on this page to avoid duplicates
    seen_figures = {}  # {figure_name: first_problem_num}
    
    # Process each problem segment
    for i, marker in enumerate(all_markers):
        start_pos = marker['end']
        end_pos = all_markers[i+1]['start'] if i+1 < len(all_markers) else len(text)
        
        segment = text[start_pos:end_pos].strip()
        
        if len(segment) < 20:
            continue
        
        # Filter out chart/graph problems
        is_chart_problem = any(re.search(pattern, segment, re.IGNORECASE) for pattern in chart_keywords)
        if is_chart_problem:
            continue
        
        # Find figures
        found_figures = []
        figure_positions = []
        
        for fig_pattern, _ in figure_patterns:
            for match in re.finditer(fig_pattern, segment, re.IGNORECASE):
                fig_name = match.group(0)
                found_figures.append(fig_name)
                figure_positions.append(match.start())
        
        # Remove duplicate figures already seen in previous problems
        unique_figures = []
        for fig in found_figures:
            if fig not in seen_figures:
                seen_figures[fig] = marker['identifier']
                unique_figures.append(fig)
        
        found_figures = unique_figures if unique_figures else found_figures
        
        # Handle merged problems
        if i+1 < len(all_markers):
            next_marker_pattern = all_markers[i+1]['identifier']
            next_match = re.search(rf'\b{re.escape(next_marker_pattern)}\b', segment)
            
            if next_match:
                next_pos = next_match.start()
                valid_figures = [fig for fig, pos in zip(found_figures, figure_positions) if pos < next_pos]
                
                if not valid_figures:
                    continue
                
                found_figures = valid_figures
                segment = segment[:next_pos].strip()
        
        # Split question and solution
        giai_match = re.search(r'\bGiải\b', segment, re.IGNORECASE)
        
        if giai_match:
            question_part = segment[:giai_match.start()].strip()
            solution_part = segment[giai_match.end():].strip()
            has_solution = True
        else:
            question_part = segment
            solution_part = ""
            has_solution = False
        
        # Clean text
        question_part = re.sub(r'^[-@•()\s=:;]+', '', question_part).strip()
        solution_part = re.sub(r'^[-@•()\s=:;]+', '', solution_part).strip()
        
        if len(question_part) < 30:
            continue
        
        question_part = re.sub(r'\s+', ' ', question_part)
        
        # === PhoBERT Analysis ===
        print(f"Analyzing {marker['identifier']} with PhoBERT...", end=" ")
        
        # Classify problem type
        prob_type, type_scores = classify_problem_type(question_part)
        
        # Analyze figure relevance
        fig_relevance = analyze_figure_relevance(question_part, found_figures)
        
        # Get embedding for similarity search later
        embedding = get_text_embedding(question_part)
        
        print(f"Type: {prob_type}, Fig relevance: {fig_relevance:.2f}")
        
        # Filter by minimum figure relevance threshold
        if fig_relevance < min_figure_relevance:
            continue
        
        results.append({
            'problem_number': marker['identifier'],
            'problem_type': marker['type'],
            'content': segment,
            'question': question_part,
            'solution': solution_part,
            'figures': list(set(found_figures)),
            'has_solution': has_solution,
            # PhoBERT features
            'ai_problem_type': prob_type,
            'type_confidence': type_scores,
            'figure_relevance': fig_relevance,
            'embedding': embedding.tolist()[0]  # Store for similarity search
        })
    
    return results

# Download and Process PDF

In [7]:
PDF_URL = "https://84864e12bc.vws.vegacdn.vn//data/doc/2025/thcslienninh/2025_2/26/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf"

pdf_file_path = download_pdf_from_url(PDF_URL, save_dir="./input")
page_images = pdf_to_images(pdf_file_path, out_dir="output/pages", dpi=300) 

PDF already exists: ./input\sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf
Converted 113 pages to images


# Process Pages with OCR

In [19]:
import gc

pages_dir = os.path.join("output", "pages")

if not os.path.exists(pages_dir):
    print(f"Not found: {pages_dir}")
else:
    # Get all page image files
    page_images = sorted([
        os.path.join(pages_dir, f) 
        for f in os.listdir(pages_dir) 
        if f.endswith('.png')
    ])
    
    print(f"Processing {len(page_images)} pages with HYBRID OCR")
    print("--------Tesseract detects layout → VietOCR recognizes Vietnamese--------\n")
    
    # OCR each page
    page_texts = {}
    
    for idx, page_img in enumerate(page_images):
        page_num = idx + 1
        print(f"[{page_num}/{len(page_images)}] {os.path.basename(page_img)}...", end=" ", flush=True)
        
        try:
            # Use Hybrid Approach (recommended)
            text = ocr_page_hybrid_tesseract_vietocr(page_img)
            page_texts[page_num] = text
            print(f"({len(text)} chars)")
            
            # Clear memory every 10 pages
            if page_num % 10 == 0:
                gc.collect()
                
        except Exception as e:
            print(f"Error: {str(e)[:100]}")
            page_texts[page_num] = ""
            gc.collect()
            continue
    
    print(f"\nCompleted OCR for {len(page_texts)} pages")
    print(f"Pages with content: {sum(1 for t in page_texts.values() if len(t) > 0)}")
    print(f"Average chars/page: {sum(len(t) for t in page_texts.values()) / len(page_texts):.0f}")


Processing 113 pages with HYBRID OCR
--------Tesseract detects layout → VietOCR recognizes Vietnamese--------

[1/113] page_001.png... (29 chars)
[2/113] page_002.png... (0 chars)
[3/113] page_003.png... (1663 chars)
[4/113] page_004.png... (923 chars)
[5/113] page_005.png... (1062 chars)
[6/113] page_006.png... 

c:\Users\Xuan Nhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\vietocr\tool\translate.py:115: RuntimeWarning: invalid value encountered in divide
  char_probs = np.sum(char_probs, axis=-1)/(char_probs>0).sum(-1)


(1079 chars)
[7/113] page_007.png... (1156 chars)
[8/113] page_008.png... (924 chars)
[9/113] page_009.png... (994 chars)
[10/113] page_010.png... (709 chars)
[11/113] page_011.png... (317 chars)
[12/113] page_012.png... (832 chars)
[13/113] page_013.png... (763 chars)
[14/113] page_014.png... (377 chars)
[15/113] page_015.png... (1096 chars)
[16/113] page_016.png... (1024 chars)
[17/113] page_017.png... (1036 chars)
[18/113] page_018.png... (1148 chars)
[19/113] page_019.png... (641 chars)
[20/113] page_020.png... (901 chars)
[21/113] page_021.png... (700 chars)
[22/113] page_022.png... (664 chars)
[23/113] page_023.png... (562 chars)
[24/113] page_024.png... (578 chars)
[25/113] page_025.png... (618 chars)
[26/113] page_026.png... (673 chars)
[27/113] page_027.png... (782 chars)
[28/113] page_028.png... (926 chars)
[29/113] page_029.png... (793 chars)
[30/113] page_030.png... (742 chars)
[31/113] page_031.png... (729 chars)
[32/113] page_032.png... (924 chars)
[33/113] page_033.png..

In [21]:
# Show OCR result for a specific page
page_to_show = 48

if page_to_show in page_texts:
    text = page_texts[page_to_show]
    print(f"PAGE {page_to_show} - OCR RESULT ({len(text)} characters)")
    print(text)
    print(f"\n{'='*70}")
    print(f"END OF PAGE {page_to_show}")
else:
    print(f"Page {page_to_show} not found in results")
    print(f"Available pages: {sorted(page_texts.keys())}")

PAGE 48 - OCR RESULT (563 characters)
4.2

4.3.

4.4.

4.5.

4.6.

48 1

Tìm độ dài X trong các hình VỀ sau (H.5.4):

M
10,5
E
x
15 F P
3
B c 24
PQ/BC
a) b)
Hình 5.4
Tìm độ dài X trong Hình 5.5:
B
6 3x
D E
3 4,5

Hình 5.5

Cho Hình 5.6.
Chứng minh rằng AB 1/Ki.

Hình 5.6

Cho hình thang ABCD (AB 11 DC). Một đường thẳng song song với hai đáy
cắt các đoạn thẳng AD; AC; BC theo thứ tự tại M, Chứng minh rằng:

AM BN CN 1
MD NC AD CB
Cho hình bình hành ABCD có M, Nlần lượt là trung điểm của AB và CD. Gọi

P Q theo thứ tự là giao điểm của và CM với đường chéo BD. Chứng
minh rằng: DP - PQ - QB.

a) b)

END OF PAGE 48


# Extract Problems with PhoBERT Analysis

In [24]:
print("Extracting problems with PhoBERT analysis...")

all_problems = []
for page_num, text in page_texts.items():
    # Extract ALL problems (min_figure_relevance=0.0), will filter later
    problems = extract_problems_with_figures_phobert(text, min_figure_relevance=0.0)
    
    for prob in problems:
        prob['page_number'] = page_num
        all_problems.append(prob)

print(f"\nTotal: {len(all_problems)} problems extracted")

# Filter by BOTH figure relevance AND actual figure presence
figure_threshold = 0.3  # Only keep problems with figure relevance >= 0.3
problems_with_figures = [
    p for p in all_problems 
    if p['figure_relevance'] >= figure_threshold 
    and len(p['figures']) > 0  # Must have at least one figure reference
]

print(f"Problems with figures (relevance >= {figure_threshold} AND has figure refs): {len(problems_with_figures)}")
print(f"Processed: {len(page_texts)} pages")

if len(problems_with_figures) > 0:
    print(f"\nPREVIEW: First 5 problems with figures")
    
    for i, prob in enumerate(problems_with_figures[:5], 1):
        print(f"[{i}] Problem {prob['problem_number']} (Page: {prob['page_number']})")
        print(f"    Type: {prob['ai_problem_type']}")
        print(f"    Figure relevance: {prob['figure_relevance']:.2f}")
        print(f"    Figures: {', '.join(prob['figures']) if prob['figures'] else 'None'}")
        print(f"    Question: {prob['question'][:150]}...")
        print()
    
    if len(problems_with_figures) > 5:
        print(f"... and {len(problems_with_figures) - 5} more problems")
    
    # Use problems_with_figures for further processing
    all_problems = problems_with_figures
else:
    print("\nNo problems found with sufficient figure relevance")

Extracting problems with PhoBERT analysis...
Analyzing 1.6 with PhoBERT... Type: word_problem, Fig relevance: 0.12
Analyzing 1.8 with PhoBERT... Type: number_theory, Fig relevance: 0.09
Analyzing 1.9 with PhoBERT... Type: number_theory, Fig relevance: 0.09
Analyzing 1.10 with PhoBERT... Type: number_theory, Fig relevance: 0.09
Analyzing 1.11 with PhoBERT... Type: number_theory, Fig relevance: 0.08
Analyzing 1.14 with PhoBERT... Type: number_theory, Fig relevance: 0.07
Analyzing 1.15 with PhoBERT... Type: algebra, Fig relevance: 0.10
Analyzing 1.16 with PhoBERT... Type: algebra, Fig relevance: 0.06
Analyzing 1.19 with PhoBERT... Type: number_theory, Fig relevance: 0.03
Analyzing 1.23 with PhoBERT... Type: word_problem, Fig relevance: 0.11
Analyzing 1.25 with PhoBERT... Type: algebra, Fig relevance: 0.11
Analyzing 1.26 with PhoBERT... Type: word_problem, Fig relevance: 0.10
Analyzing 1.28 with PhoBERT... Type: number_theory, Fig relevance: 0.09
Analyzing 1.30 with PhoBERT... Type: algebr

In [25]:
if len(problems_with_figures) > 0:
    print(f"\nPREVIEW problems with figures")
    
    for i, prob in enumerate(problems_with_figures[:10], 1):
        print(f"[{i}] Problem {prob['problem_number']} (Page: {prob['page_number']})")
        print(f"    Type: {prob['ai_problem_type']}")
        print(f"    Figure relevance: {prob['figure_relevance']:.2f}")
        print(f"    Figures: {', '.join(prob['figures']) if prob['figures'] else 'None'}")
        print(f"    Question: {prob['question'][:150]}...")
        print()
    
    if len(problems_with_figures) > 5:
        print(f"... and {len(problems_with_figures) - 5} more problems")
    
    # Use problems_with_figures for further processing
    all_problems = problems_with_figures
else:
    print("\nNo problems found with sufficient figure relevance")


PREVIEW problems with figures
[1] Problem 2.12 (Page: 24)
    Type: number_theory
    Figure relevance: 0.39
    Figures: (H.2.3), Hình 2.3
    Question: Từ một khối lập phương CÓ độ dài cạnh là X 4 3 (cm), ta cắt bỏ một khối lập phương có độ dài X 1(cm) (H.2.3). Tính thể tích phần còn lại, viết kết quả...

[2] Problem 2.24 (Page: 30)
    Type: word_problem
    Figure relevance: 0.40
    Figures: (H.2.4), Hình 2.4
    Question: Từ một miếng bìa có dạng hình tròn (H.2.4) với bán kinh R(cm), người ta khoét một hình tròn Ở giữa có bán kính 5 R. a) Viết công thức tính diện tích p...

[3] Problem 4.6 (Page: 48)
    Type: algebra
    Figure relevance: 0.82
    Figures: Hình 5.6, Hình 5.5, Hình 5.4, (H.5.4)
    Question: 48 1 Tìm độ dài X trong các hình VỀ sau (H.5.4): M 10,5 E x 15 F P 3 B c 24 PQ/BC a) b) Hình 5.4 Tìm độ dài X trong Hình 5.5: B 6 3x D E 3 4,5 Hình 5....

[4] Problem 5.19 (Page: 71)
    Type: algebra
    Figure relevance: 0.40
    Figures: Hình 5.12
    Question: Tính Biểu 

# Save Results with PhoBERT Features

In [23]:
output_folder = os.path.join("..", "dataset", "output", "mapped_result")
os.makedirs(output_folder, exist_ok=True)

# Prepare data structure
output_data = {
    "metadata": {
        "pdf_path": pdf_file_path,
        "total_pages": len(page_texts),
        "total_problems": len(all_problems),
        "problems_with_solution": sum(1 for p in all_problems if p.get('has_solution', False)),
        "problems_without_solution": sum(1 for p in all_problems if not p.get('has_solution', False)),
        "processed_at": datetime.now().isoformat(),
        "processing_method": "PhoBERT + PaddleOCR + VietOCR",
        "model_info": {
            "phobert": "vinai/phobert-base",
            "vietocr": "vgg_transformer",
            "paddleocr": "vi"
        }
    },
    "problems": []
}

# Save problems with PhoBERT features
for i, prob in enumerate(all_problems, 1):
    output_data["problems"].append({
        "id": i,
        "problem_number": prob['problem_number'],
        "page_number": prob.get('page_number'),
        "question": prob['question'],
        "solution": prob.get('solution', ''),
        "figure_references": prob['figures'],
        "has_solution": prob.get('has_solution', False),
        
        # PhoBERT analysis
        "ai_features": {
            "problem_type": prob.get('ai_problem_type'),
            "type_confidence": prob.get('type_confidence', {}),
            "figure_relevance": prob.get('figure_relevance', 0),
            "embedding_preview": prob.get('embedding', [])[:10]  # First 10 dimensions
        }
    })

# Save main results
output_file = os.path.join(output_folder, "problems_with_figures_phobert.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ RESULTS SAVED")
print(f"={'='*60}")
print(f"Folder: {output_folder}")
print(f"File: problems_with_figures_phobert.json")
print(f"\nStatistics:")
print(f"  Total problems: {output_data['metadata']['total_problems']}")
print(f"  ├─ With solution: {output_data['metadata']['problems_with_solution']}")
print(f"  └─ Without solution: {output_data['metadata']['problems_without_solution']}")
print(f"  Total pages: {output_data['metadata']['total_pages']}")
print(f"\n  Features: PhoBERT embeddings, problem classification, figure relevance")
print(f"={'='*60}")


✅ RESULTS SAVED
Folder: ..\dataset\output\mapped_result
File: problems_with_figures_phobert.json

Statistics:
  Total problems: 25
  ├─ With solution: 1
  └─ Without solution: 24
  Total pages: 113

  Features: PhoBERT embeddings, problem classification, figure relevance


# Similarity Search Demo

In [ ]:
def find_similar_problems(query_text, problems, top_k=5):
    """
    Find similar problems using PhoBERT embeddings
    """
    query_emb = get_text_embedding(query_text)
    
    similarities = []
    for prob in problems:
        prob_emb = np.array(prob['embedding']).reshape(1, -1)
        similarity = cosine_similarity(query_emb, prob_emb)[0][0]
        similarities.append({
            'problem': prob,
            'similarity': float(similarity)
        })
    
    similarities.sort(key=lambda x: x['similarity'], reverse=True)
    return similarities[:top_k]


# Demo: Find problems similar to a query
if len(all_problems) > 0:
    print("\n🔍 SIMILARITY SEARCH DEMO")
    print("="*60)
    
    query = "Tính diện tích tam giác dựa vào hình vẽ"
    print(f"Query: {query}\n")
    
    similar = find_similar_problems(query, all_problems, top_k=3)
    
    for i, item in enumerate(similar, 1):
        prob = item['problem']
        sim = item['similarity']
        print(f"[{i}] Similarity: {sim:.3f}")
        print(f"    Problem: {prob['problem_number']} (Page {prob['page_number']})")
        print(f"    Type: {prob['ai_problem_type']}")
        print(f"    Question: {prob['question'][:150]}...")
        print()
else:
    print("\n⚠️ No problems found for similarity search demo")